# Document RAG Assistant - Part 1: Data Preparation

**Goal of this notebook:** take a raw PDF and turn it into clean, overlapping text chunks
that are ready to be embedded and stored in a vector database.

**Our source document:** `NovaTech_Company_Policy_Handbook.pdf` - a 10-page company policy
handbook covering leave, remote work, expenses, information security, conduct, product
warranty, returns and customer support. It is a realistic corporate PDF, which means it comes
with realistic problems: a running header on every page, tables, bullet lists and sentences
broken across lines. This notebook uses that one document and nothing else.

This notebook builds the **complete 7-step RAG pipeline** in two halves:

| # | Step | Where |
|---|------|-------|
| 1 | **Document Loading** - read the PDF, keep page numbers | Part 1 |
| 2 | **Preprocessing** - clean the text with NLP techniques | Part 1 |
| 3 | **Chunking** - sliding window with overlap | Part 1 |
| 4 | **Embedding** - text to vectors | Part 2 |
| 5 | **Vector storage** - ChromaDB | Part 2 |
| 6 | **Retrieval** - cosine similarity, top 3 | Part 2 |
| 7 | **Generation** - LLM answers from our data only | Part 2 |

**Part 1 starts immediately below. Part 2 continues in the same notebook**, picking up the
chunks Part 1 produces - run the cells top to bottom and the whole pipeline builds itself.

**Why these three steps matter so much:** *garbage in, garbage out*. If the text is dirty, or a
chunk cuts a sentence in half, the retrieval step will hand the LLM the wrong context - and no
amount of prompt engineering can fix that later.

**No frameworks used** - no LangChain, no LlamaIndex. Just `pypdf`, `nltk`, `re` and plain
Python, so you can see exactly what happens at every line.

---
## Step 0: Setup

We need two libraries:

* **pypdf** - opens a PDF and pulls the text out, page by page.
* **nltk** (Natural Language Toolkit) - the classic NLP library. We use it for sentence
  splitting, word tokenization, stop-word removal and lemmatization.

`nltk` also needs a few small data files (the sentence model, the stop-word list, and the
dictionary used by the lemmatizer). We download them once below.

In [ ]:
# Installs into THIS kernel. Safe to re-run; skip if already installed.
%pip install -q pypdf nltk

import os           # file paths
import re           # regular expressions, used for text cleaning
import json         # to save our final chunks
import unicodedata  # to fix weird encoding characters

from pypdf import PdfReader

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# --- Download the small NLTK data packs (silent, runs once, cached afterwards) ---
for pack in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pack, quiet=True)

# Create the NLP tools we will reuse throughout the notebook
STOP_WORDS = set(stopwords.words("english"))  # ~180 common words: the, is, are, for ...
LEMMATIZER = WordNetLemmatizer()              # running -> run, policies -> policy

print("Setup complete.")
print(f"Stop-word list loaded: {len(STOP_WORDS)} words (first 10: {sorted(STOP_WORDS)[:10]})")

---
## Step 1: Document Loading

The first job is to get raw text out of the PDF.

Two things come out of a loader, and **both** are important:

* **content** - the actual text on the page.
* **metadata** - where the text came from (file name, page number). Without metadata the
  final answer cannot cite its source, and you can never check whether the AI is telling
  the truth.

We store each page as a simple Python dictionary:

```python
{"content": "text of the page ...", "metadata": {"source": "file.pdf", "page": 1}}
```

We load **page by page** (not the whole file as one blob) so the page number stays attached
to the text.

In [ ]:
def load_pdf(file_path):
    """
    Read a PDF and return one dictionary per page.

    Args:
        file_path (str): path to the .pdf file

    Returns:
        list[dict]: [{"content": str, "metadata": {"source": str, "page": int}}, ...]
    """
    reader = PdfReader(file_path)
    file_name = os.path.basename(file_path)
    pages = []

    # enumerate(..., start=1) so page numbers read the way a human would say them
    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""   # extract_text() returns None on image-only pages

        # Skip pages that are empty or almost empty (covers, blank pages, pure images)
        if len(text.strip()) < 30:
            continue

        pages.append({
            "content": text,
            "metadata": {"source": file_name, "page": page_number}
        })

    return pages

In [ ]:
# ---------------------------------------------------------------
# Our one and only source document. It should sit in the same
# folder as this notebook.
# ---------------------------------------------------------------
PDF_PATH = "NovaTech_Company_Policy_Handbook.pdf"

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(
        f"Could not find '{PDF_PATH}'.\n"
        f"Place the policy handbook in this folder: {os.getcwd()}"
    )

documents = load_pdf(PDF_PATH)

# ----------------------------- inspect what we loaded -----------------------------
print("=" * 70)
print(f"Total documents (pages) loaded: {len(documents)}")
print("=" * 70)

first = documents[0]
print(f"\nMetadata of document 1 : {first['metadata']}")
print(f"Characters on this page: {len(first['content'])}")
print("\n--- First 500 characters (RAW, straight out of the PDF) ---")
print(repr(first["content"][:500]))   # repr() reveals the hidden \n and \xa0 characters

---
## Step 2: Text Preprocessing (Cleaning)

Look at the `repr()` output above - the raw text is full of `\n` line breaks in the middle of
sentences, double spaces, bullet glyphs and non-breaking spaces (`\xa0`). A PDF stores
*visual layout*, not *sentences*, so extracted text always needs repair.

There is also a second, sneakier problem. Scroll the raw output above and you will see
these four lines:

```
NovaTech Systems Pvt. Ltd.
Employee & Customer Policy Handbook  |  v4.2
Confidential - Internal & Customer Distribution
Page 1
```

That is the running **header and footer**, and it appears on *every single page*. If we leave
it in, every chunk we create starts with the same boilerplate. That hurts twice: we pay to
embed and store the same sentence 10 times, and because every chunk shares that text, they
all look slightly more similar to each other, which blurs retrieval.

So cleaning happens in **two passes**:

1. `remove_boilerplate()` - a **corpus-level** pass. It looks across all pages, finds lines
   that repeat on most of them, and deletes them. This has to run *first*, while the line
   breaks are still intact.
2. `clean_text()` - a **page-level** pass that fixes six specific problems:

| # | Problem | Fix |
|---|---------|-----|
| 1 | Weird unicode (`\u2019`, `\xa0`, ligatures like `fi`) | Unicode normalisation (NFKC) + replacements |
| 2 | Words split across lines: `prepro-\ncessing` | Re-join the hyphenated word |
| 3 | A single `\n` inside a sentence | Turn it into a space |
| 4 | Real paragraph breaks (`\n\n`) | Keep them - chunking needs them |
| 5 | Bullets, stray page numbers, control characters | Delete |
| 6 | Runs of spaces and tabs | Collapse to one space |

**What about lowercasing?** The session notes list it as optional. We keep the original case in
the text we store, because capital letters help the embedding model recognise names like
*NovaTech* or *Pulse Pro*. Lowercasing is applied only inside the NLP analysis functions below,
where it genuinely helps.

In [ ]:
def find_boilerplate_lines(docs, min_share=0.6):
    """
    Find lines that repeat across most pages - i.e. running headers and footers.

    We do NOT hard-code "NovaTech Systems Pvt. Ltd." here. The whole point is that this
    works on any PDF: whatever line shows up on 60%+ of the pages is, by definition,
    furniture rather than content.

    Args:
        docs (list[dict]) : the loaded (still raw) documents
        min_share (float) : fraction of pages a line must appear on to count as boilerplate

    Returns:
        set[str]: the lines to delete
    """
    counts = {}
    for doc in docs:
        # set() so a line repeated twice on one page still counts as one page
        for line in {l.strip() for l in doc["content"].split("\n")}:
            # Only short lines can be furniture. A 300-character line is a paragraph.
            if 3 <= len(line) <= 120:
                counts[line] = counts.get(line, 0) + 1

    threshold = max(2, int(len(docs) * min_share))
    return {line for line, count in counts.items() if count >= threshold}


def remove_boilerplate(text, boilerplate):
    """
    Delete repeated header/footer lines and standalone page numbers.

    Args:
        text (str)            : raw page text
        boilerplate (set[str]): output of find_boilerplate_lines()

    Returns:
        str: the page text without its furniture
    """
    kept = []
    for line in text.split("\n"):
        stripped = line.strip()

        if stripped in boilerplate:          # a known running header/footer
            continue
        # A line that is nothing but a page number: "7", "Page 7", "7 of 10"
        if re.fullmatch(r"(page\s*)?\d{1,3}(\s*(of|/)\s*\d{1,3})?", stripped, flags=re.I):
            continue

        kept.append(line)

    return "\n".join(kept)


# --- Find the furniture in OUR document and show what we caught ---
boilerplate = find_boilerplate_lines(documents)

print(f"Scanned {len(documents)} pages.")
print(f"Found {len(boilerplate)} repeating line(s) to strip:\n")
for line in sorted(boilerplate):
    print(f"  - {line!r}")

In [ ]:
def clean_text(text):
    """
    Clean raw PDF text while preserving paragraph structure.

    Args:
        text (str): raw text from the PDF

    Returns:
        str: cleaned text, paragraphs separated by a blank line
    """
    # --- 1. Fix encoding issues --------------------------------------------------
    # NFKC turns look-alike characters into their plain equivalent,
    # e.g. the ligature "fi" -> "fi", and a full-width space -> a normal space.
    text = unicodedata.normalize("NFKC", text)

    replacements = {
        "\u2019": "'", "\u2018": "'",      # curly single quotes -> '
        "\u201c": '"', "\u201d": '"',      # curly double quotes -> "
        "\u2013": "-", "\u2014": "-",      # en dash / em dash   -> -
        "\u2026": "...",                   # ellipsis character
        "\xa0":   " ",                     # non-breaking space
        "\ufffd": "",                      # the "unknown character" diamond
    }
    for bad, good in replacements.items():
        text = text.replace(bad, good)

    # --- 2. Re-join words that were hyphenated across a line break ----------------
    # "prepro-\ncessing"  ->  "preprocessing"
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)

    # --- 3 & 4. Protect real paragraph breaks, flatten the rest -------------------
    # A blank line means "new paragraph" -> remember it with a placeholder.
    text = re.sub(r"\n\s*\n+", "<<<PARA>>>", text)
    # Any remaining single newline is just PDF line wrapping -> becomes a space.
    text = text.replace("\n", " ")
    # Put the real paragraph breaks back.
    text = text.replace("<<<PARA>>>", "\n\n")

    # --- 5. Remove junk characters ------------------------------------------------
    text = re.sub(r"[\u2022\u25cf\u25aa\u00b7]", " ", text)   # bullet glyphs
    text = re.sub(r"[^\x20-\x7E\n]", " ", text)               # non-printable leftovers
    text = re.sub(r"\n\s*\d{1,3}\s*\n", "\n", text)           # stray page numbers on own line

    # --- 6. Collapse whitespace ---------------------------------------------------
    text = re.sub(r"[ \t]+", " ", text)        # many spaces -> one space
    text = re.sub(r" ?\n ?", "\n", text)       # no spaces hugging a newline
    text = re.sub(r"\n{3,}", "\n\n", text)     # never more than one blank line

    return text.strip()

In [ ]:
# ----------------- Apply BOTH passes to every page, keeping metadata -----------------
stripped_documents = []     # after pass 1 only - kept so we can verify the pass worked
cleaned_documents = []      # after both passes - this is what we chunk

for doc in documents:
    stripped = remove_boilerplate(doc["content"], boilerplate)   # pass 1: corpus-level
    stripped_documents.append(stripped)

    cleaned_documents.append({
        "content": clean_text(stripped),                         # pass 2: page-level
        "metadata": doc["metadata"]      # metadata travels with the text, untouched
    })

# ----------------------------- BEFORE vs AFTER -----------------------------
before = documents[0]["content"][:400]
after = cleaned_documents[0]["content"][:400]

print("=" * 70)
print("BEFORE CLEANING (raw - note the \\n line breaks and odd characters)")
print("=" * 70)
print(repr(before))

print("\n" + "=" * 70)
print("AFTER CLEANING (readable sentences, paragraphs preserved)")
print("=" * 70)
print(after)

# ----------------------------- What actually changed -----------------------------
raw_all = "".join(d["content"] for d in documents)
clean_all = "".join(d["content"] for d in cleaned_documents)
removed = len(raw_all) - len(clean_all)

print("\n" + "-" * 70)
print(f"Characters before cleaning  : {len(raw_all):,}")
print(f"Characters after cleaning   : {len(clean_all):,}")
print(f"Boilerplate/noise removed   : {removed:,} characters "
      f"({removed / max(len(raw_all), 1) * 100:.1f}%)")
print(f"Line breaks before          : {raw_all.count(chr(10)):,}  "
      f"(most of these sit INSIDE sentences)")
print(f"Line breaks after           : {clean_all.count(chr(10)):,}  "
      f"(only real paragraph breaks are left)")
print(f"Sentences recovered intact  : {len(sent_tokenize(clean_all)):,}")

# Prove the header is really gone.
# This has to be a WHOLE-LINE check, not a substring check: the cover page genuinely
# says "NovaTech Systems Pvt. Ltd." in the middle of a sentence, and that text is
# content, not furniture. We only ever deleted lines that matched exactly.
deleted = sum(1 for d in documents for line in d["content"].split("\n")
              if line.strip() in boilerplate)
leftover = sum(1 for text in stripped_documents for line in text.split("\n")
               if line.strip() in boilerplate)
print(f"\nBoilerplate lines deleted : {deleted} across {len(documents)} pages")
print(f"Boilerplate lines left     : {leftover}  (we want 0)")

print("\nTwo different wins here:")
print("  - DELETION: the repeated header/footer is gone, so we never embed it again.")
print("  - REPAIR  : the character count barely moves for the rest, but the text went")
print("              from 'lines on a page' to 'readable sentences'.")

---
## Step 2b: NLP Techniques - Tokenization, Stop Words, Lemmatization

Cleaning removed the *formatting* noise. NLP preprocessing goes one level deeper and works on
the *language* itself. Three classic techniques:

| Technique | What it does | Why it helps RAG |
|-----------|--------------|------------------|
| **Tokenization** | Splits text into sentences or words | Lets us measure size and cost; sentences are the unit we chunk on |
| **Stop-word removal** | Drops low-meaning words (*is, the, are, for*) | Fewer tokens sent to the LLM = lower API cost |
| **Lemmatization** | Reduces words to their root (*running -> run*, *policies -> policy*) | A query about "policy" now matches text that says "policies" |

### An important judgement call

Applying all of this to the text you *store* would destroy readability:
`"20 day paid annual leave full time employee"` is cheap, but hard for a human to verify and
awkward for a modern embedding model.

So we do what production pipelines do:

* **Store the cleaned, natural text** in the chunks (Step 3).
* **Use the NLP functions as an analysis layer** - to count tokens, estimate cost, build a
  keyword view, and understand how much noise a document carries.

The functions below are written so you can switch them on for the stored text too
(`normalize_for_search`), and the cell after them measures exactly what that would save you.

In [ ]:
def split_into_sentences(text):
    """Tokenize text into a list of sentences using NLTK's trained sentence model."""
    return sent_tokenize(text)


def split_into_words(text):
    """Tokenize text into a list of word tokens (punctuation counts as its own token)."""
    return word_tokenize(text)


def remove_stop_words(text):
    """
    Drop common English stop words and punctuation.

    "The policy for sick leave is 12 days" -> "policy sick leave 12 days"
    """
    tokens = word_tokenize(text.lower())
    kept = [t for t in tokens if t.isalnum() and t not in STOP_WORDS]
    return " ".join(kept)


def lemmatize(text):
    """
    Reduce every word to its dictionary root form.

    "The developers are running better systems" -> "the developer are running better system"
    (NLTK lemmatizes nouns by default, which is what we want for document search.)
    """
    tokens = word_tokenize(text.lower())
    roots = [LEMMATIZER.lemmatize(t) for t in tokens if t.isalnum()]
    return " ".join(roots)


def normalize_for_search(text):
    """
    Full NLP pipeline: lowercase -> lemmatize -> remove stop words.

    Use this for keyword matching and cost estimation. We deliberately do NOT use it
    on the chunk text we store, because it strips readability.
    """
    return remove_stop_words(lemmatize(text))


def estimate_tokens(text):
    """
    Rough token count without calling any API.

    Rule of thumb for English and GPT-style tokenizers: ~4 characters per token.
    Good enough to compare two versions of the same text and to estimate cost.
    """
    return max(1, round(len(text) / 4))

In [ ]:
# ------------------------ Demonstration on a real paragraph ------------------------
demo = split_into_sentences(cleaned_documents[0]["content"])
demo_text = " ".join(demo[:4])          # take the first 4 sentences of page 1

print("=" * 70)
print("1. SENTENCE TOKENIZATION")
print("=" * 70)
for i, sentence in enumerate(demo[:4], start=1):
    print(f"  [{i}] {sentence}")

print("\n" + "=" * 70)
print("2. WORD TOKENIZATION")
print("=" * 70)
words = split_into_words(demo_text)
print(f"  Total word tokens: {len(words)}")
print(f"  First 20 tokens  : {words[:20]}")

print("\n" + "=" * 70)
print("3. LEMMATIZATION (word -> root form)")
print("=" * 70)
example = "The company updated its leave policies and replaced the laptop batteries."
print(f"  Original   : {example}")
print(f"  Lemmatized : {lemmatize(example)}")
# Show a few individual mappings so the effect is obvious
print("\n  Word -> root:")
for w in ["policies", "batteries", "queries", "analyses", "employees"]:
    print(f"    {w:<12} -> {LEMMATIZER.lemmatize(w)}")
print("\n  Why this matters: a user asking about 'policy' now matches a document")
print("  that says 'policies' - the two become the same token.")
print("\n  Note: NLTK assumes every word is a NOUN unless you tell it otherwise, so")
print(f"  'running' stays 'running' ({LEMMATIZER.lemmatize('running')}) until you pass "
      "pos='v':")
print(f"    running (as a verb) -> {LEMMATIZER.lemmatize('running', pos='v')}")

print("\n" + "=" * 70)
print("4. STOP-WORD REMOVAL")
print("=" * 70)
print(f"  Before : {demo_text[:200]}")
print(f"  After  : {remove_stop_words(demo_text)[:200]}")

print("\n" + "=" * 70)
print("5. COST IMPACT (why anyone bothers)")
print("=" * 70)
full_tokens = estimate_tokens(demo_text)
slim_tokens = estimate_tokens(normalize_for_search(demo_text))
print(f"  Natural text      : ~{full_tokens} tokens")
print(f"  Fully normalized  : ~{slim_tokens} tokens")
print(f"  Reduction         : {(full_tokens - slim_tokens) / full_tokens * 100:.0f}% fewer tokens")
print("\n  -> A real saving, but readability is gone. We keep natural text in the chunks")
print("     and use this normalized view only for analysis and keyword matching.")

---
## Step 3: Chunking with a Sliding Window

An LLM cannot read a 500-page document in one prompt, so we split the text into **chunks**.
How we split decides whether retrieval works at all.

### The problem with fixed-size chunking

Cut strictly every 500 characters and you eventually cut mid-sentence:

```
Chunk A: "... The return policy is"
Chunk B: "30 days from the date of delivery ..."
```

Neither chunk answers *"How long is the return policy?"*. The information exists in the
document but is unreachable.

### The fix: a sliding window with overlap

Each new chunk starts a little *before* the previous one ended, so the end of chunk A is
repeated at the start of chunk B:

```
|---------- chunk 1 (500 chars) ----------|
                          |---------- chunk 2 (500 chars) ----------|
                          |<- overlap ->|
                             100 chars
```

The start pointer moves forward by `chunk_size - overlap` each time. Any sentence that got cut
at a boundary now appears **complete** in at least one chunk.

We build two versions:

1. `sliding_window_chunk()` - the textbook character-based version.
2. `sentence_aware_sliding_window()` - the same idea, but boundaries snap to sentence ends.
   This is our default, because it gives **contextually accurate** chunks: no chunk ever
   starts or ends mid-sentence.

In [ ]:
def sliding_window_chunk(text, chunk_size=800, overlap=150):
    """
    Classic character-based sliding window.

    Args:
        text (str)      : the text to split
        chunk_size (int): characters per chunk
        overlap (int)   : characters repeated from the end of the previous chunk

    Returns:
        list[str]: the chunks
    """
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size, or the window never moves")

    chunks = []
    start = 0
    step = chunk_size - overlap        # how far the window slides each time

    while start < len(text):
        chunk = text[start:start + chunk_size]   # take a slice
        if chunk.strip():                        # ignore whitespace-only slices
            chunks.append(chunk.strip())
        if start + chunk_size >= len(text):      # we reached the end
            break
        start += step                            # slide the window forward

    return chunks

In [ ]:
# ---------------- See the overlap with your own eyes ----------------
demo_doc = cleaned_documents[0]["content"]
demo_chunks = sliding_window_chunk(demo_doc, chunk_size=500, overlap=100)

print(f"Page 1 is {len(demo_doc)} characters -> {len(demo_chunks)} chunks "
      f"(size=500, overlap=100)\n")

print("=" * 70)
print("LAST 100 CHARACTERS OF CHUNK 1")
print("=" * 70)
print(demo_chunks[0][-100:])

print("\n" + "=" * 70)
print("FIRST 100 CHARACTERS OF CHUNK 2  <- identical text, that IS the overlap")
print("=" * 70)
print(demo_chunks[1][:100])

print("\n" + "-" * 70)
print("Notice chunk 1 still ends mid-sentence. The overlap rescues the meaning,")
print("but we can do better by snapping the boundaries to sentence ends.")

In [ ]:
def sentence_aware_sliding_window(text, chunk_size=800, overlap=150):
    """
    Sliding window that never cuts a sentence in half.  <-- our default strategy

    How it works:
      1. Split the text into sentences with NLTK.
      2. Pack whole sentences into a chunk until adding one more would exceed chunk_size.
      3. For the next chunk, step BACK over the last few sentences until we have at least
         `overlap` characters of repeated text, and start the new chunk there.

    A sentence longer than chunk_size (rare - usually a table row or a long list) is split
    with the plain character window, so nothing is ever lost.

    Args:
        text (str)      : cleaned text
        chunk_size (int): target maximum characters per chunk
        overlap (int)   : minimum characters of context repeated between neighbours

    Returns:
        list[str]: the chunks
    """
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    # --- 1. Build the list of "units" we are allowed to move around ----------------
    units = []
    for sentence in sent_tokenize(text):
        sentence = sentence.strip()
        if not sentence:
            continue
        if len(sentence) <= chunk_size:
            units.append(sentence)
        else:
            # Oversized sentence: fall back to the character window for this one only
            units.extend(sliding_window_chunk(sentence, chunk_size, overlap))

    if not units:
        return []

    chunks = []
    i = 0                                  # index of the first sentence in the current chunk

    while i < len(units):
        # --- 2. Greedily pack whole sentences into this chunk -----------------------
        current, length, j = [], 0, i
        while j < len(units) and length + len(units[j]) + 1 <= chunk_size:
            current.append(units[j])
            length += len(units[j]) + 1    # +1 for the space we will join with
            j += 1

        if not current:                    # safety net: always make progress
            current, j = [units[i]], i + 1

        chunks.append(" ".join(current))

        if j >= len(units):                # that was the last chunk
            break

        # --- 3. Step back over sentences to create the overlap ----------------------
        back_chars, k = 0, j
        while k > i + 1 and back_chars < overlap:
            k -= 1
            back_chars += len(units[k]) + 1

        i = k                              # k is always > i, so the loop cannot stall

    return chunks

In [ ]:
# ---------------- Compare the two strategies on the same page ----------------
fixed_style = sliding_window_chunk(demo_doc, chunk_size=800, overlap=150)
sentence_style = sentence_aware_sliding_window(demo_doc, chunk_size=800, overlap=150)

print(f"Character sliding window : {len(fixed_style)} chunks")
print(f"Sentence-aware window    : {len(sentence_style)} chunks\n")

print("=" * 70)
print("CHARACTER WINDOW - end of chunk 1 (cut wherever the 800th character landed)")
print("=" * 70)
print("..." + fixed_style[0][-160:])

print("\n" + "=" * 70)
print("SENTENCE-AWARE WINDOW - end of chunk 1 (ends on a full stop)")
print("=" * 70)
print("..." + sentence_style[0][-160:])

print("\n" + "=" * 70)
print("SENTENCE-AWARE WINDOW - start of chunk 2 (repeats the previous sentences)")
print("=" * 70)
print(sentence_style[1][:160] + "...")

---
## Step 4: Build the Final Chunk List with Metadata

Now we run the chunker over **every page** and wrap each chunk in a dictionary.

Each chunk carries:

```python
{
  "content":  "the chunk text ...",
  "metadata": {
      "source":   "NovaTech_Company_Policy_Handbook.pdf",  # which file
      "page":     3,                                       # which page
      "chunk_id": 7,                                       # unique id across the run
      "char_count": 782,
      "token_estimate": 196
  }
}
```

That metadata is what lets the finished assistant say *"according to page 3 of the policy
document..."* instead of just asserting an answer.

### Choosing chunk_size and overlap

* **chunk_size = 800 characters** (~200 tokens). Big enough to hold a complete idea,
  small enough that a retrieved chunk is mostly signal.
* **overlap = 150 characters** (~19%). The common rule of thumb is 10-20% of chunk size:
  enough to bridge a sentence boundary, not so much that you pay to store the same text twice.

In [ ]:
# -------------------------- Configuration (tune these) --------------------------
CHUNK_SIZE = 800    # target characters per chunk  (~200 tokens)
OVERLAP = 150       # characters repeated between neighbouring chunks (~19%)


def build_chunks(docs, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    """
    Turn a list of cleaned documents into a flat list of chunk dictionaries.

    Args:
        docs (list[dict]): output of the cleaning step
        chunk_size (int) : characters per chunk
        overlap (int)    : characters of overlap

    Returns:
        list[dict]: chunks with content + metadata
    """
    all_chunks = []
    chunk_id = 0                                   # unique id across the whole document set

    for doc in docs:
        pieces = sentence_aware_sliding_window(doc["content"], chunk_size, overlap)

        for piece in pieces:
            all_chunks.append({
                "content": piece,
                "metadata": {
                    # copy the page's metadata, then add chunk-level details
                    **doc["metadata"],
                    "chunk_id": chunk_id,
                    "char_count": len(piece),
                    "token_estimate": estimate_tokens(piece),
                }
            })
            chunk_id += 1

    return all_chunks


chunks = build_chunks(cleaned_documents)

print("=" * 70)
print("CHUNKING COMPLETE")
print("=" * 70)
print(f"Pages in            : {len(cleaned_documents)}")
print(f"Chunks out          : {len(chunks)}")
print(f"chunk_size / overlap: {CHUNK_SIZE} / {OVERLAP} characters")

print("\n--- SAMPLE CHUNK (the 2nd one) ---")
sample = chunks[1]
print(f"Metadata: {sample['metadata']}")
print(f"\nContent:\n{sample['content']}")

---
## Step 5: Quality Checks

Never trust a data-prep script you have not inspected. Four checks:

1. **Size distribution** - are chunks roughly the size we asked for?
2. **Overlap actually present** - do neighbouring chunks really share text?
3. **No broken sentences** - does every chunk end on proper punctuation?
4. **Nothing lost** - is all the original text still represented?

In [ ]:
sizes = [c["metadata"]["char_count"] for c in chunks]

print("=" * 70)
print("CHECK 1 - CHUNK SIZE DISTRIBUTION")
print("=" * 70)
print(f"  Smallest : {min(sizes)} chars")
print(f"  Largest  : {max(sizes)} chars   (must be <= {CHUNK_SIZE})")
print(f"  Average  : {sum(sizes) / len(sizes):.0f} chars")
print(f"  Total    : {sum(sizes):,} chars across {len(chunks)} chunks")
print(f"  Est. tokens to embed: ~{sum(c['metadata']['token_estimate'] for c in chunks):,}")


print("\n" + "=" * 70)
print("CHECK 2 - IS THE OVERLAP REALLY THERE?")
print("=" * 70)


def shared_tail(a, b):
    """Length of the longest text that is both the end of `a` and the start of `b`."""
    limit = min(len(a), len(b))
    for size in range(limit, 0, -1):
        if a[-size:] == b[:size]:
            return size
    return 0


pairs_checked, pairs_with_overlap = 0, 0
for first_chunk, second_chunk in zip(chunks, chunks[1:]):
    # Only compare chunks from the same page - a page boundary has no overlap by design.
    if first_chunk["metadata"]["page"] != second_chunk["metadata"]["page"]:
        continue
    pairs_checked += 1
    if shared_tail(first_chunk["content"], second_chunk["content"]) > 0:
        pairs_with_overlap += 1

print(f"  Neighbouring pairs on the same page : {pairs_checked}")
print(f"  Pairs that share text               : {pairs_with_overlap}")
if pairs_checked:
    print(f"  Overlap coverage                    : "
          f"{pairs_with_overlap / pairs_checked * 100:.0f}%")


print("\n" + "=" * 70)
print("CHECK 3 - DO CHUNKS END ON A COMPLETE SENTENCE?")
print("=" * 70)
complete = sum(1 for c in chunks if c["content"].rstrip().endswith((".", "!", "?", ":", '"')))
print(f"  Chunks ending on punctuation : {complete} / {len(chunks)} "
      f"({complete / len(chunks) * 100:.0f}%)")
print("  (A few exceptions are normal: table rows and bullet lists have no full stop.)")


print("\n" + "=" * 70)
print("CHECK 4 - DID WE LOSE ANY TEXT?")
print("=" * 70)
source_chars = sum(len(d["content"]) for d in cleaned_documents)
chunk_chars = sum(sizes)
print(f"  Cleaned source : {source_chars:,} chars")
print(f"  Sum of chunks  : {chunk_chars:,} chars")
print(f"  Ratio          : {chunk_chars / source_chars:.2f}x")
print("  (Expect slightly MORE than 1.0x - the overlap is stored twice on purpose.)")

---
## Step 6: Save the Prepared Data

The chunks go into a JSON file. Part 2 of the project can load this file directly and jump
straight to embeddings - no need to re-parse the PDF every time.

In [ ]:
OUTPUT_PATH = "prepared_chunks.json"

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

print(f"Saved {len(chunks)} chunks to '{OUTPUT_PATH}'")
print(f"File size: {os.path.getsize(OUTPUT_PATH) / 1024:.1f} KB")

# Reload it to prove the file is valid and complete
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    reloaded = json.load(f)

print(f"\nReload test: {len(reloaded)} chunks read back successfully.")
print(f"First chunk metadata: {reloaded[0]['metadata']}")
print(f"\nPreview:\n{reloaded[0]['content'][:300]}...")

---
## Summary - What This Script Does

```
raw PDF
   |
   |-- load_pdf()                      page by page, metadata attached
   |
   |-- remove_boilerplate()            strips the running header/footer that
   |                                   repeats on every page
   |
   |-- clean_text()                    unicode fixes, de-hyphenation, whitespace,
   |                                   junk removal, paragraphs preserved
   |
   |-- NLP layer                       sentence + word tokenization,
   |   (analysis / cost)               stop-word removal, lemmatization
   |
   |-- sentence_aware_sliding_window() overlapping chunks that never cut a sentence
   |
   `-- prepared_chunks.json            ready for embedding (Part 2)
```

### Key decisions and the reasoning

| Decision | Value | Why |
|----------|-------|-----|
| Loading granularity | one document per page | keeps page numbers for citations |
| Boilerplate removal | detected, not hard-coded | works on any PDF, not just this one |
| Lowercasing stored text | no | capitals carry meaning for embeddings (*NovaTech*, *Pulse Pro*) |
| Stop-word removal on stored text | no | saves tokens but hurts readability and verification |
| NLP functions | kept as an analysis layer | measure cost, build keyword views, switch on when needed |
| Chunk strategy | sentence-aware sliding window | overlap preserves context, sentence snapping prevents mid-sentence cuts |
| chunk_size | 800 chars (~200 tokens) | one complete idea per chunk |
| overlap | 150 chars (~19%) | the standard 10-20% rule of thumb |

### Libraries used

`pypdf` (PDF extraction), `nltk` (tokenization, stop words, lemmatization),
`re` and `unicodedata` (cleaning), `json` (output). No RAG framework.

### What we produced

`prepared_chunks.json` - every chunk of the NovaTech policy handbook, cleaned, overlapped,
and tagged with its source file and page number.

The best way to sanity-check the output is to open that file and read a few chunks. If a
human can answer a policy question from a single chunk, the embedding model will be able to
find it too.

### Next: Part 2, below

Part 1 is done - the data is prepared. Part 2 starts in the next cell and uses the `chunks`
list we just built: it embeds each chunk, stores the vectors in ChromaDB, retrieves the top 3
for a question, and has an LLM answer from those chunks alone.

---
---
# Part 2: The End-to-End RAG Pipeline

Part 1 left us with 36 clean, overlapping chunks of the NovaTech policy handbook, each
tagged with its page number. Right now that is just a list of strings - we can read it, but
we cannot *search* it by meaning.

Part 2 completes the remaining four steps:

| # | Step | What we build |
|---|------|---------------|
| 4 | **Embedding** | Turn every chunk into a 384-number vector that captures its meaning |
| 5 | **Vector storage** | Load those vectors into ChromaDB, a database built for meaning-search |
| 6 | **Retrieval** | Take a user question and find the top 3 most relevant chunks |
| 7 | **Generation** | Hand those 3 chunks to an LLM and get a grounded answer |

By the end, you will be able to ask *"How many days of annual leave do I get?"* and get an
answer that comes **only** from the handbook - with the page number it came from.

### Still no frameworks

We keep the Part 1 rule: no LangChain, no LlamaIndex. Four new libraries, each doing one
visible job:

* **sentence-transformers** - runs the embedding model on your own machine. Free, no API
  key, nothing leaves your laptop.
* **chromadb** - the vector database.
* **numpy** - to show the cosine similarity maths by hand, once.
* **requests** - to call the LLM's REST API directly, so you can see exactly what is sent.

---
## Step 7: Setup for Part 2

The embedding model (`all-MiniLM-L6-v2`) is about 90 MB. It downloads once on the first run
and is cached after that, so the first execution of the next cell is slower than later ones.

In [ ]:
# Installs into THIS kernel. Safe to re-run; skip if already installed.
%pip install -q sentence-transformers chromadb numpy requests

import logging
import textwrap

import numpy as np
import requests
import chromadb
from sentence_transformers import SentenceTransformer

# These libraries are chatty on import. Quieten them so the notebook output stays readable.
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

# --- Load the embedding model (downloads ~90 MB the first time, then cached) ---
print("Loading embedding model 'all-MiniLM-L6-v2' ...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Model loaded. Output dimensions: "
      f"{embedding_model.get_sentence_embedding_dimension()}")

# Safety check: Part 2 builds directly on the `chunks` list from Part 1.
print(f"\nChunks carried over from Part 1: {len(chunks)}")
print(f"Source document: {chunks[0]['metadata']['source']}")

---
## Step 8: Embeddings - Turning Text Into Numbers

A computer cannot compare two sentences for *meaning*. It can only compare numbers. An
**embedding model** solves this: it reads a piece of text and outputs a list of numbers (a
**vector**) that represents what the text is *about*.

The useful property is this: **texts with similar meanings get similar vectors**, even when
they share no words at all. "Sick leave" and "medical time off" land close together. "Sick
leave" and "laptop warranty" land far apart.

Our model, `all-MiniLM-L6-v2`, outputs **384 numbers** per text. It runs locally, costs
nothing, and your document never leaves your machine - which matters when the document is a
confidential company handbook.

> **Using OpenAI instead:** replace `embed_texts()` with a call to
> `openai.embeddings.create(model="text-embedding-3-small", input=texts)`. That model gives
> 1536 dimensions and slightly better accuracy, but it costs money and sends your text to a
> third party. Everything downstream in this notebook works the same either way.

In [ ]:
def embed_texts(texts):
    """
    Convert a list of strings into a list of 384-number vectors.

    Args:
        texts (list[str]): the texts to embed

    Returns:
        list[list[float]]: one vector per input text
    """
    # normalize_embeddings=True scales every vector to length 1. That makes cosine
    # similarity equal to a simple dot product, which is faster and keeps scores in
    # a predictable range.
    vectors = embedding_model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return vectors.tolist()


def cosine_similarity(vec_a, vec_b):
    """
    Measure how similar two vectors are, from -1 (opposite) to 1 (identical).

    Cosine similarity looks at the ANGLE between two vectors, not their length.
    Two texts pointing in the same direction mean the same thing.
    """
    a, b = np.array(vec_a), np.array(vec_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

In [ ]:
# ---------- Prove that embeddings capture MEANING, not keywords ----------
pairs = [
    ("sick leave",            "medical time off"),      # same idea, zero shared words
    ("sick leave",            "annual holiday"),         # related, but different
    ("sick leave",            "laptop battery warranty"),  # unrelated
]

print("=" * 70)
print("SEMANTIC SIMILARITY: do related meanings score higher?")
print("=" * 70)
for text_a, text_b in pairs:
    vec_a, vec_b = embed_texts([text_a, text_b])
    score = cosine_similarity(vec_a, vec_b)
    bar = "#" * int(score * 40)
    print(f"  {score:5.3f} |{bar:<40}| '{text_a}'  vs  '{text_b}'")

print("\n  'sick leave' and 'medical time off' share NO words, yet score highest.")
print("  That is the whole point: this is search by meaning, not by keyword.")

In [ ]:
# ---------- Embed all 36 chunks ----------
chunk_texts = [c["content"] for c in chunks]

print(f"Embedding {len(chunk_texts)} chunks ...")
chunk_embeddings = embed_texts(chunk_texts)

print(f"Done.\n")
print(f"  Chunks embedded        : {len(chunk_embeddings)}")
print(f"  Numbers per chunk      : {len(chunk_embeddings[0])}")
print(f"  Total numbers stored   : {len(chunk_embeddings) * len(chunk_embeddings[0]):,}")
print(f"\n  First 8 numbers of chunk 0:")
print(f"  {[round(x, 4) for x in chunk_embeddings[0][:8]]}")
print(f"\nEach chunk is now a point in 384-dimensional space.")

---
## Step 9: The Vector Database

We could keep the vectors in a Python list and compare the query against all 36 by hand. That
works for 36 chunks. It does not work for 36,000.

A **vector database** is built for exactly one job: given a query vector, find the nearest
stored vectors, fast. We use **ChromaDB** because it runs in-process with no server to set up.

Three things go in together for every chunk:

| What | Why |
|------|-----|
| `embeddings` | the 384 numbers - what gets searched |
| `documents` | the original readable text - what we show the LLM |
| `metadatas` | source file, page, chunk_id - what lets us cite the answer |

**One setting that matters:** we create the collection with `hnsw:space = "cosine"`. ChromaDB
defaults to squared L2 distance; we want cosine, the metric the whole course is built around.

In [ ]:
# --- Create an in-memory ChromaDB client ---
# Use chromadb.PersistentClient(path="./chroma_db") instead if you want the database
# to survive after the notebook shuts down.
chroma_client = chromadb.Client()

COLLECTION_NAME = "document_rag"

# Re-running this cell should not fail, so drop any previous version first.
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass   # nothing to delete on the first run

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},   # use cosine similarity, not the L2 default
)

# --- Load every chunk into the database ---
collection.add(
    ids=[f"chunk_{c['metadata']['chunk_id']}" for c in chunks],   # unique string IDs
    embeddings=chunk_embeddings,                                  # the searchable vectors
    documents=chunk_texts,                                        # the readable text
    metadatas=[c["metadata"] for c in chunks],                    # source + page for citing
)

print(f"Stored {collection.count()} chunks in ChromaDB collection '{COLLECTION_NAME}'.")
print(f"Distance metric: cosine")

# Verify one record came back exactly as we put it in
check = collection.get(ids=["chunk_0"], include=["documents", "metadatas"])
print(f"\nSpot-check of 'chunk_0':")
print(f"  Metadata: {check['metadatas'][0]}")
print(f"  Text    : {check['documents'][0][:120]}...")

---
## Step 10: Retrieval - Finding the Right 3 Chunks

Now the search itself. When a question arrives:

1. Embed the **question** with the **same model** used for the chunks. This is essential - two
   different models produce vectors that live in different spaces and cannot be compared.
2. Ask ChromaDB for the nearest stored vectors by cosine similarity.
3. Keep the **top 3**.

ChromaDB returns a *distance*, where smaller means closer. For cosine space,
`similarity = 1 - distance`, so a similarity of 1.0 is a perfect match and 0.0 is unrelated.

### The score threshold, and why it matters

Here is the trap: a vector database **always** returns its *k* nearest neighbours. Ask
*"Who won the cricket world cup?"* and it will still hand you 3 policy chunks - they will just
be terrible matches, scoring around 0.09 instead of 0.70.

If we pass those to the LLM anyway, we invite a hallucination. So we add
`MIN_SIMILARITY`: any chunk scoring below it is dropped. If nothing survives, we never call
the LLM at all - we just say we do not know. This is the first of two defences against
answering questions our document cannot answer.

In [ ]:
TOP_K = 3                # how many chunks to retrieve - the deliverable asks for 3
MIN_SIMILARITY = 0.25    # drop anything less relevant than this


def retrieve(query, n_results=TOP_K, min_similarity=MIN_SIMILARITY):
    """
    Find the chunks most relevant to a question.

    Args:
        query (str)            : the user's question
        n_results (int)        : how many chunks to fetch
        min_similarity (float) : chunks scoring below this are discarded

    Returns:
        list[dict]: [{"content", "metadata", "similarity"}, ...], best match first
    """
    # 1. Embed the question using the SAME model that embedded the chunks
    query_embedding = embed_texts([query])

    # 2. Ask the database for the nearest vectors
    response = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )

    # 3. Unpack the result and convert distance -> similarity.
    #    Chroma nests everything one level deep (one list per query), hence the [0].
    results = []
    for text, meta, distance in zip(response["documents"][0],
                                    response["metadatas"][0],
                                    response["distances"][0]):
        similarity = 1 - distance
        if similarity >= min_similarity:       # 4. drop the weak matches
            results.append({
                "content": text,
                "metadata": meta,
                "similarity": similarity,
            })

    return results


def show_retrieval(query):
    """Print what retrieve() found for a question - a debugging helper."""
    found = retrieve(query)
    print("=" * 74)
    print(f"QUERY: {query}")
    print("=" * 74)

    if not found:
        print("  No chunk scored above the threshold - this question is out of scope.")
        return

    for rank, item in enumerate(found, start=1):
        meta = item["metadata"]
        print(f"\n  [{rank}] similarity {item['similarity']:.3f}  "
              f"| page {meta['page']} | chunk {meta['chunk_id']}")
        snippet = textwrap.shorten(item["content"], width=260, placeholder=" ...")
        for line in textwrap.wrap(snippet, width=70):
            print(f"      {line}")

In [ ]:
# ---------- Test retrieval on questions of different kinds ----------
show_retrieval("How many days of paid annual leave do employees get?")
print()
show_retrieval("What should I do if my work laptop is stolen?")
print()
show_retrieval("Who won the cricket world cup in 2011?")   # deliberately out of scope

---
## Step 11: Building the Prompt

Retrieval found the right text. Now we have to make the LLM use it - *and only it*.

An LLM will happily answer from its own training data if you let it. The prompt is where we
take that option away. Ours has four parts:

1. **Role** - who the assistant is.
2. **Constraint** - the critical line: answer only from the context, and if the answer is not
   there, say so. This is our second defence against hallucination (the score threshold was
   the first).
3. **Context** - the retrieved chunks, numbered `[1] [2] [3]` with their page numbers, so the
   model can cite them.
4. **Question** - what the user actually asked.

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful assistant for NovaTech Systems employees and customers. "
    "Answer questions ONLY using the context provided below. "
    "If the answer is not in the context, reply exactly: "
    "'I don't have enough information to answer this.' "
    "Never use outside knowledge. Never guess. "
    "Cite the source of each fact using its bracket number, like [1] or [2]."
)


def build_prompt(query, retrieved):
    """
    Assemble the user-side prompt from the question and the retrieved chunks.

    Args:
        query (str)       : the user's question
        retrieved (list)  : output of retrieve()

    Returns:
        str: the prompt to send to the LLM
    """
    # Number each chunk and label it with its page, so the model can cite properly
    context_blocks = []
    for i, item in enumerate(retrieved, start=1):
        page = item["metadata"]["page"]
        context_blocks.append(f"[{i}] (page {page}) {item['content']}")

    context = "\n\n".join(context_blocks)

    return (
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        f"Answer based only on the context above. Cite your sources as [1], [2], etc. "
        f"If the context does not contain the answer, say "
        f"\"I don't have enough information to answer this.\""
    )


# ---------- Show one complete prompt, exactly as the LLM will receive it ----------
demo_query = "How much notice do I need to give for 5 days of leave?"
demo_chunks = retrieve(demo_query)
demo_prompt = build_prompt(demo_query, demo_chunks)

print("=" * 74)
print("SYSTEM MESSAGE")
print("=" * 74)
for line in textwrap.wrap(SYSTEM_PROMPT, width=74):
    print(line)

print("\n" + "=" * 74)
print("USER MESSAGE (this is the full prompt, nothing hidden)")
print("=" * 74)
print(demo_prompt)

---
## Step 12: Calling the LLM

The last piece. We send the system message and the prompt to a chat model and read back the
answer.

### Getting a free API key

**Google Gemini** has a free tier with no billing required - the easiest option:

1. Go to [aistudio.google.com](https://aistudio.google.com) and sign in.
2. Click **Get API key** then **Create API key in new project**.
3. Paste it into `API_KEY` in the next cell.

**OpenAI** works too - get a key at `platform.openai.com/api-keys` and set
`LLM_PROVIDER = "openai"`. OpenAI requires billing to be set up.

> **Security:** never commit an API key to GitHub or share a notebook with a key still in it.
> The safer habit is to set it as an environment variable and read it with
> `os.environ.get("GEMINI_API_KEY")`, which is what the cell below does if you leave
> `API_KEY` blank.

### Running without a key

If you have not set a key, the pipeline still runs end to end - retrieval, scoring, prompt
building, all of it. In place of a generated answer you get the top retrieved chunk, clearly
marked so there is no confusing it for LLM output. Add a key whenever you are ready.

In [ ]:
# ----------------------------- LLM configuration -----------------------------
LLM_PROVIDER = "gemini"          # "gemini" or "openai"
API_KEY = ""                     # paste your key here, or leave blank to use an env var

MODEL_NAME = {
    "gemini": "gemini-3.5-flash",   # see the note below if this stops working
    "openai": "gpt-4o-mini",
}[LLM_PROVIDER]

# If the first choice is overloaded (HTTP 503), try these in order before giving up.
# The "lite" models are smaller and far less contended, so they almost always answer.
FALLBACK_MODELS = {
    "gemini": ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite"],
    "openai": [],
}[LLM_PROVIDER]

TEMPERATURE = 0.2                # low = factual and repeatable, high = creative

# Fall back to an environment variable if API_KEY was left empty
if not API_KEY:
    env_var = "GEMINI_API_KEY" if LLM_PROVIDER == "gemini" else "OPENAI_API_KEY"
    API_KEY = os.environ.get(env_var, "")

HAS_API_KEY = bool(API_KEY)

if HAS_API_KEY:
    print(f"API key found. Using {LLM_PROVIDER} / {MODEL_NAME} at temperature {TEMPERATURE}.")
else:
    print("No API key set - running in retrieval-only mode.")
    print("Everything except the final generation step still works.")
    print("Set API_KEY above to turn on real answers.")

In [ ]:
import time

MAX_RETRIES = 3          # popular free models get busy; retrying usually fixes it
REQUEST_TIMEOUT = 60     # seconds to wait for one response
TIME_BUDGET = 90         # ceiling on ONE question, across retries and models

# Set to True the first time the API reports an exhausted quota. Every later call
# then returns instantly instead of waiting on a request that is certain to fail.
QUOTA_EXHAUSTED = False


def post_with_retry(url, headers, payload, deadline):
    """
    POST to an API, retrying only when retrying can actually help.

    Three kinds of failure, treated differently:

      * 503 (overloaded) or a timeout / connection drop - TEMPORARY. The server was
        fine a second ago and will be fine again, so we wait and try once more.
      * 429 (quota or rate limit) - DO NOT retry here. If you have exhausted a daily
        free-tier quota, hammering it for another minute changes nothing; it only
        makes the notebook appear to hang. We return immediately and let the caller
        decide whether a different model has quota left.
      * anything else (400, 401, 404) - PERMANENT. A bad key or a retired model name
        will never fix itself.

    `deadline` is an absolute time.monotonic() value. We stop retrying once it passes,
    so one question can never run away with the whole notebook.

    Returns:
        requests.Response: the final response, successful or not

    Raises:
        requests.RequestException: if every attempt failed to connect
    """
    response = None      # the last response we actually received
    last_error = None    # the last connection failure, if we never got one

    for attempt in range(MAX_RETRIES):
        try:
            remaining = deadline - time.monotonic()
            if remaining <= 0:
                break
            response = requests.post(url, headers=headers, json=payload,
                                     timeout=min(REQUEST_TIMEOUT, remaining))

            if response.status_code != 503:
                return response           # success, 429, or a permanent error
            reason = "model busy (503)"

        except (requests.Timeout, requests.ConnectionError) as error:
            # No status code to inspect - the response never arrived.
            last_error = error
            reason = type(error).__name__

        wait = 4 * (attempt + 1)
        if attempt < MAX_RETRIES - 1 and time.monotonic() + wait < deadline:
            print(f"    ({reason} - retrying in {wait}s, "
                  f"attempt {attempt + 2} of {MAX_RETRIES})")
            time.sleep(wait)
        else:
            break

    # Out of retries or out of time. Hand back the last response if we got one,
    # otherwise re-raise the connection error.
    if response is not None:
        return response
    if last_error is not None:
        raise last_error
    raise requests.Timeout(f"No response within the {TIME_BUDGET}s time budget")


def call_llm(system_prompt, user_prompt):
    """
    Send a prompt to the configured LLM and return its text reply.

    We call the REST API directly with `requests` rather than using an SDK, so you can
    see exactly what goes over the wire.

    Args:
        system_prompt (str): the role and constraints
        user_prompt (str)  : the context and the question

    Returns:
        str: the model's answer, or an error message
    """
    # Once the daily quota is gone it is gone. Skip the call rather than waiting
    # ~3 minutes to be told so again. (See try_one_model for why it takes so long.)
    if QUOTA_EXHAUSTED:
        return ("[Skipped: the free-tier quota is already exhausted for this session. "
                "It resets daily - or set up billing to continue now.]")

    # One clock for the whole question, however many models we end up trying
    deadline = time.monotonic() + TIME_BUDGET

    for model in [MODEL_NAME] + FALLBACK_MODELS:
        answer, try_another = try_one_model(model, system_prompt, user_prompt, deadline)

        if not try_another:
            return answer                      # success, or a permanent error

        if time.monotonic() >= deadline:
            return f"{answer}\n[Gave up after {TIME_BUDGET}s.]"

        print(f"    ({model} unavailable - trying the next model)")

    return answer   # every model failed; return the last error message


def try_one_model(model, system_prompt, user_prompt, deadline):
    """
    Make one LLM call against a specific model.

    Returns:
        tuple[str, bool]: (the answer or an error message, is_another_model_worth_trying)
    """
    try:
        if LLM_PROVIDER == "gemini":
            url = (f"https://generativelanguage.googleapis.com/v1beta/models/"
                   f"{model}:generateContent")
            payload = {
                # Gemini has no separate "system" role, so it goes in system_instruction
                "system_instruction": {"parts": [{"text": system_prompt}]},
                "contents": [{"parts": [{"text": user_prompt}]}],
                "generationConfig": {"temperature": TEMPERATURE},
            }
            response = post_with_retry(
                url,
                {"x-goog-api-key": API_KEY, "Content-Type": "application/json"},
                payload,
                deadline,
            )
            response.raise_for_status()
            data = response.json()

            # Gemini can return a candidate with NO text - e.g. when it stops early
            # or a safety filter trips. Handle that instead of crashing on a KeyError.
            try:
                text = data["candidates"][0]["content"]["parts"][0]["text"].strip()
                return text, False
            except (KeyError, IndexError):
                reason = data.get("candidates", [{}])[0].get("finishReason", "unknown")
                return (f"[No text returned. finishReason={reason}. "
                        f"Raw response: {json.dumps(data)[:300]}]"), False

        else:   # openai
            response = post_with_retry(
                "https://api.openai.com/v1/chat/completions",
                {"Authorization": f"Bearer {API_KEY}",
                 "Content-Type": "application/json"},
                {
                    "model": model,
                    "messages": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt},
                    ],
                    "temperature": TEMPERATURE,
                },
                deadline,
            )
            response.raise_for_status()
            return response.json()["choices"][0]["message"]["content"].strip(), False

    except requests.HTTPError as error:
        global QUOTA_EXHAUSTED

        # Show the server's own message - it says exactly what is wrong
        status = error.response.status_code

        if status == 429:
            # Measured behaviour: once the free-tier quota is spent, Google stalls
            # each request for ~3 minutes before returning the 429. The `timeout`
            # argument does NOT help - it limits the gap BETWEEN bytes, not the
            # total call, so a trickling response resets it forever. The only
            # effective defence is to stop calling.
            QUOTA_EXHAUSTED = True
        try:
            detail = error.response.json()["error"]["message"]
        except Exception:
            detail = error.response.text[:200]
        # Only 503 ("this model is busy") is worth trying another model for.
        # 429 is a QUOTA error and the free-tier quota is shared across models -
        # measured: when one model returns 429, every other model does too. Trying
        # them anyway just makes the notebook hang for a minute before the same
        # failure. Everything else (400/401/404) is permanent.
        return f"[API error {status}: {detail}]", (status == 503)

    except Exception as error:
        # Never let an API hiccup crash the notebook
        return f"[LLM call failed: {type(error).__name__}: {error}]", False


def retrieval_only_answer(retrieved):
    """
    Used when no key is set, or when the key stopped working (quota, outage).

    This is NOT a generated answer - it is the single best-matching chunk, printed
    verbatim. It is labelled loudly so it can never be mistaken for LLM output.
    """
    best = retrieved[0]
    return (f"[NO LLM ANSWER - this is the top retrieved chunk, printed verbatim. "
            f"Retrieval worked; only the generation step is unavailable.]\n\n"
            f"(page {best['metadata']['page']}, similarity {best['similarity']:.3f})\n"
            f"{best['content']}")

### Check your key before going further

The next cell makes the smallest possible call - two words in, a few words out - so that if
anything is wrong with your key, you find out here rather than halfway through the demo.
Common failures and what they mean:

| Message | Cause |
|---------|-------|
| `API error 400: API key not valid` | the key is wrong or has a stray space |
| `API error 404: ... no longer available` | that model has been retired - see below |
| `API error 429: ... quota` | free-tier rate limit hit; wait a minute |
| `API error 503: ... high demand` | the model is busy; `post_with_retry` handles this |
| `[LLM call failed: ConnectionError]` | no internet, or a firewall is blocking the call |

### When a model name stops working

Google retires model names regularly, and a notebook written a few months ago will hit a
404 saying the model *"is no longer available"*. This is the single most likely reason this
notebook breaks later. The fix takes a minute: ask your key which models it can actually
use, then set `MODEL_NAME` to one of them.

```python
import requests
response = requests.get(
    "https://generativelanguage.googleapis.com/v1beta/models",
    headers={"x-goog-api-key": API_KEY},
)
for model in response.json()["models"]:
    if "generateContent" in model.get("supportedGenerationMethods", []):
        print(model["name"].replace("models/", ""))
```

Pick any `*-flash` model from that list - flash models are the fast, cheap tier and are what
the free quota is designed around. If the newest one returns 503 because everybody is using
it, an older flash release is usually wide open.

In [ ]:
# ---------------- Smallest possible end-to-end test of the LLM connection ----------------
if HAS_API_KEY:
    print(f"Testing {LLM_PROVIDER} / {MODEL_NAME} ...\n")
    reply = call_llm(
        "You are a test endpoint. Reply with exactly one word.",
        "Reply with the single word: connected",
    )
    print(f"  Model replied: {reply!r}")

    if reply.startswith("["):
        print("\n  Something went wrong - see the message above.")
        print("  The rest of the notebook will still run, but answers will not generate.")
        HAS_API_KEY = False       # fall back to retrieval-only for the demo below
    else:
        print("\n  Connection works. Real answers are switched on.")
else:
    print("No API key configured - skipping the connection test.")
    print("The pipeline below will run in retrieval-only mode.")

---
## Step 13: The Complete Pipeline

One function that runs the whole thing: question in, grounded answer out.

```
query -> embed -> search ChromaDB -> top 3 chunks -> build prompt -> LLM -> answer
```

Notice the early exit. If retrieval finds nothing above the threshold, we return
*"I don't have enough information to answer this."* **without calling the LLM**. The model
never even sees the question, so it cannot invent an answer. That saves an API call and
removes the hallucination risk entirely.

In [ ]:
def rag_pipeline(query, n_results=TOP_K, verbose=True):
    """
    Run the full RAG pipeline for one question.

    Steps: embed the query -> retrieve top chunks -> build the prompt -> call the LLM.

    Args:
        query (str)     : the user's question
        n_results (int) : how many chunks to retrieve
        verbose (bool)  : print the intermediate steps

    Returns:
        dict: {"query", "answer", "chunks", "sources"}
    """
    # --- Step 1: retrieve ---
    retrieved = retrieve(query, n_results=n_results)

    if verbose:
        print("=" * 74)
        print(f"QUERY: {query}")
        print("=" * 74)
        print(f"Retrieved {len(retrieved)} relevant chunk(s) "
              f"above the {MIN_SIMILARITY} similarity threshold.")

    # --- Early exit: nothing relevant, so do not call the LLM at all ---
    if not retrieved:
        answer = "I don't have enough information to answer this."
        if verbose:
            print("No chunk cleared the threshold, so the LLM was never called.")
            print(f"\nANSWER:\n  {answer}\n")
        return {"query": query, "answer": answer, "chunks": [], "sources": []}

    # --- Report what we found ---
    sources = []
    for i, item in enumerate(retrieved, start=1):
        meta = item["metadata"]
        label = f"{meta['source']} p.{meta['page']}"
        sources.append(label)
        if verbose:
            print(f"   [{i}] similarity {item['similarity']:.3f}  |  {label}")

    # --- Step 2: build the prompt ---
    prompt = build_prompt(query, retrieved)

    # --- Step 3: generate ---
    if HAS_API_KEY:
        answer = call_llm(SYSTEM_PROMPT, prompt)
    else:
        answer = retrieval_only_answer(retrieved)

    if verbose:
        print("\nANSWER:")
        for paragraph in answer.split("\n"):
            for line in textwrap.wrap(paragraph, width=70) or [""]:
                print(f"  {line}")
        print()

    return {"query": query, "answer": answer, "chunks": retrieved, "sources": sources}

---
## Step 14: Live Demo

Six questions, each testing something different:

1. **Straightforward** - the answer sits in one section.
2. **Spans two sections** - the answer is split across the handbook.
3. **Semantic, not keyword** - the question uses words the document never contains.
4. **A trap** - several sections have similar-looking numbers; does it pick the right one?
5. **Obviously out of scope** - unrelated vocabulary. The *threshold* should catch this.
6. **Plausibly out of scope** - sounds exactly like a NovaTech policy question, but the
   handbook says nothing about it. The threshold will **not** catch this one.

Questions 5 and 6 are the interesting pair, and they fail in different ways on purpose.

Question 5 uses words nowhere near the handbook, so every chunk scores around 0.09 and the
threshold blocks it before the LLM is ever called.

Question 6 is the harder case. "What is NovaTech's policy on pet insurance for employees?"
is built from the handbook's own vocabulary - *NovaTech*, *policy*, *employees* - so the
nearest chunks score around 0.58 and sail straight through the threshold. Retrieval looks
confident and is completely wrong. Nothing about the *score* can save us here; only the
prompt instruction can, by telling the model to refuse when the retrieved text does not
actually contain the answer.

That is the honest reason this pipeline needs both defences rather than either one.

In [ ]:
test_questions = [
    # 1. Straightforward lookup
    ("Straightforward",
     "How many days of paid annual leave do full-time employees get?"),

    # 2. Spans two sections (remote-work allowance on p.3 + device rules on p.5)
    ("Spans two sections",
     "I work remotely and my company laptop was stolen. What do I need to do, "
     "and what equipment am I allowed to claim?"),

    # 3. Semantic match - the handbook never uses the phrase "work from home"
    ("Semantic, not keyword",
     "What is the policy on working from home?"),

    # 4. The trap - several sections mention "30 days" and "7 to 10 business days"
    ("Ambiguous numbers",
     "How long does a refund take to reach my bank account?"),

    # 5. Out of scope, unrelated words -> the THRESHOLD should block this
    ("Out of scope (obvious)",
     "Who won the cricket world cup in 2011?"),

    # 6. Out of scope, handbook vocabulary -> only the PROMPT can block this
    ("Out of scope (plausible)",
     "What is NovaTech's policy on pet insurance for employees?"),
]

results = []
for number, (label, question) in enumerate(test_questions, start=1):
    print(f"\n{'#' * 74}")
    print(f"# TEST {number} of {len(test_questions)}  -  {label}")
    print(f"{'#' * 74}")
    results.append((label, rag_pipeline(question)))

In [ ]:
# ---------------- What the six tests actually showed ----------------
print("=" * 74)
print("RESULTS AT A GLANCE")
print("=" * 74)
print(f"{'Test':<26} | {'Chunks':>6} | {'Top score':>9} | Pages used")
print("-" * 74)

for label, result in results:
    chunks_found = len(result["chunks"])
    top = f"{result['chunks'][0]['similarity']:.3f}" if chunks_found else "  -  "
    pages = sorted({c["metadata"]["page"] for c in result["chunks"]}) or ["none"]
    print(f"{label:<26} | {chunks_found:>6} | {top:>9} | {pages}")

print("\n" + "=" * 74)
print("THREE THINGS WORTH NOTICING")
print("=" * 74)
print("""
1. THE THRESHOLD WORKS - BUT ONLY ON OBVIOUS MISSES.
   The cricket question scored so low that nothing survived filtering, so the
   LLM was never called. Cost: zero. Hallucination risk: zero.

2. THE THRESHOLD IS NOT ENOUGH.
   The pet-insurance question scored ~0.58 and passed straight through, because
   it is phrased in the handbook's own vocabulary. Retrieval was confident and
   wrong. Only the prompt instruction can catch this one - which is why the
   system prompt spells out the exact refusal wording.

3. TOP-3 CAN BE TOO NARROW FOR MULTI-PART QUESTIONS.
   Test 2 asked two things at once (what to do + what to claim). All three
   retrieved chunks came from page 5, the device-security section; the home
   office allowance on page 3 never made the cut. The fix is not a better
   prompt - it is retrieving more chunks and re-ranking them, or splitting
   compound questions before retrieval.
""")

---
## Step 15: Summary - The Complete Pipeline

```
PART 1 - Data Preparation
   raw PDF
      |-- load_pdf()                       10 pages, metadata attached
      |-- remove_boilerplate()             strip the repeating header/footer
      |-- clean_text()                     unicode, de-hyphenation, whitespace
      |-- sentence_aware_sliding_window()  36 overlapping chunks
      `-- prepared_chunks.json

PART 2 - Retrieval and Generation
      |-- embed_texts()                    36 chunks -> 36 vectors of 384 numbers
      |-- ChromaDB collection              cosine-space vector index
      |-- retrieve()                       question -> top 3 chunks above threshold
      |-- build_prompt()                   numbered context + strict instruction
      `-- call_llm()                       grounded answer with [1][2] citations
```

### Every decision, and why

| Decision | Value | Why |
|----------|-------|-----|
| Embedding model | `all-MiniLM-L6-v2` (384-dim) | free, local, the handbook never leaves the machine |
| Vector store | ChromaDB, in-process | no server to run; swap in FAISS for millions of vectors |
| Distance metric | cosine | compares meaning (angle), ignores text length |
| Top-K | 3 | enough context to answer, few enough to stay focused |
| Similarity threshold | 0.25 | off-topic questions score ~0.09, real ones 0.39-0.70 |
| Temperature | 0.2 | we want the same factual answer every time, not creativity |
| Prompt constraint | "answer ONLY from context" | the instruction that prevents hallucination |

### The two defences against hallucination

This is the part worth remembering. Grounding an LLM is not one trick, it is two:

1. **The similarity threshold** stops a bad question before it reaches the model. *"Who won
   the cricket world cup?"* scores ~0.09 against every chunk, nothing survives filtering, and
   the LLM is never called. Zero cost, zero risk.
2. **The prompt constraint** handles the case the threshold cannot see. *"What is NovaTech's
   policy on pet insurance?"* scores ~0.58, because it is built from the handbook's own
   words - *NovaTech*, *policy*, *employees*. Retrieval is confident and wrong. No score
   tells you that; only the instruction to refuse when the context lacks the answer does.

The demo above shows both cases firing. This is the real lesson of the notebook: a
similarity score measures whether text *looks* related, never whether it actually answers
the question.

### What RAG did and did not do

We never trained or fine-tuned anything. The LLM learned nothing about NovaTech. We simply
handed it the right page at the right moment - the open-book exam, not the revision course.
Change a policy in the PDF, re-run the notebook, and the answers change immediately.

### Where to take it next

* **Persistence** - `chromadb.PersistentClient(path="./chroma_db")` so you embed once, not
  on every run.
* **More documents** - the metadata already carries `source`, so chunks from a second PDF
  slot in with no code changes.
* **Re-ranking** - retrieve 10 chunks, then use a cross-encoder to pick the best 3.
* **Evaluation** - write 20 questions with known answers and measure how often the right
  chunk is retrieved. That number, not intuition, tells you whether a change helped.